# MDST 3D Generative Pipeline — Week 1

## Goal for this week

Build the very first version of our procedural city generator, from scratch:

1. **Flat terrain** (no hills, water, or archetypes yet — that comes later)
2. **Basic building placement** — one downtown center, density-based
3. **Green space** — parks filling in empty ground
4. A **preview** showing terrain height, building height, and land use side by side

By the end of this notebook, running all cells should produce a city that looks
something like a simple downtown surrounded by open ground and scattered parks.

**How this notebook works:** most cells have a function with a `# TODO` and a
`YOUR CODE HERE` line for you to fill in. Read the docstring and the hints in
the comments — they tell you exactly what the line should compute. After each
one, run the "check yourself" cell right below it — it'll tell you if your
implementation looks right before you move on.

Don't worry about making this fast or elegant. Getting it *correct* and
*understood* is the whole point this week — we'll build on this every week
after.

## Setup

Nothing to fill in here — just imports and a couple of constants we'll reuse throughout.

In [57]:
import numpy as np
import matplotlib.pyplot as plt

# grid size: our city is a SIZE x SIZE grid of cells. 128 is a good default --
# big enough to look like a real city, small enough to be fast to work with.
SIZE = 128

# land_use codes -- every cell in the grid is exactly one of these.
# (We'll add ROAD as a 4th code in a future week -- not yet!)
LAND_EMPTY = 0
LAND_BUILDING = 1
LAND_GREEN = 2

print(f"Grid size: {SIZE}x{SIZE} = {SIZE*SIZE} cells")

Grid size: 128x128 = 16384 cells


## Step 1: Terrain (flat)

Every city in our generator sits on a terrain **heightmap** — a `SIZE x SIZE`
array of elevation values. This week we only build the `flat` terrain type, so
this is the simplest possible version: mostly level ground with a *little*
gentle texture so it doesn't look robotically perfect.

Later weeks will add hills, mountains, rivers, coastlines, and more — but the
heightmap idea stays exactly the same. Getting this right now means later
weeks are just "swap in a fancier formula," not "start over."


In [60]:
def generate_flat_terrain(size, seed=None):
    """
    Returns a (size, size) float32 array of terrain heights, roughly flat
    but with a little gentle random texture.

    TODO: create terrain_height as small random noise centered near zero.

    Hint:
        rng.normal(size=(size, size)) gives you random values roughly in
        the range -3 to 3. We want our terrain to stay CLOSE to flat, so
        scale that noise down a lot -- try multiplying by something small
        like 0.05.
    """
    rng = np.random.default_rng(seed)

    terrain_height = rng.normal(size=(size, size)) * 0.05

    return terrain_height.astype(np.float32)

In [62]:
# --- check yourself ---
_test = generate_flat_terrain(SIZE, seed=0)
assert _test.shape == (SIZE, SIZE), f"expected shape ({SIZE},{SIZE}), got {_test.shape}"
assert _test.dtype == np.float32, f"expected float32, got {_test.dtype}"
assert abs(_test.mean()) < 0.5, "terrain should stay close to flat (mean near 0) -- is your noise scaled down enough?"
assert _test.std() > 0.001, "terrain has no texture at all -- did you forget to add the noise?"
print("Looks good! terrain_height mean =", round(_test.mean(), 4), " std =", round(_test.std(), 4))

Looks good! terrain_height mean = 0.0003  std = 0.0498


## Step 2: Building placement

Now let's place buildings. This week's version: **one downtown center**, with
buildings more likely the closer a cell is to that center (a "monocentric"
layout, in the language we'll use in later weeks).

The core idea:
1. Compute how far every cell is from the center point.
2. Turn that distance into an **intensity** — high near the center, decaying
   outward.
3. For each cell, roll a random number; if it's less than
   `intensity * density`, that cell becomes a building.

This distance-based-probability trick is the same idea we'll reuse for city
archetypes, clustering, and more in later weeks — you're building a real,
reusable piece of the final pipeline right now, not throwaway code.


In [65]:
def generate_buildings(size, density=0.5, seed=None):
    """
    Returns (building_height, land_use), each a (size, size) array.
    building_height is 0.0 where there's no building.
    land_use uses LAND_EMPTY / LAND_BUILDING from above.

    TODO: fill in `intensity` and `is_building` below.
    """
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size]

    # the single downtown center, right in the middle of the grid
    cx, cy = size / 2, size / 2

    # distance from every cell to the center, normalized so it's roughly 0-1
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2) / size

    # TODO: turn distance into an intensity that's HIGH near the center
    # (dist=0) and decays toward 0 far away.
    # Hint: np.exp(-dist * k) does exactly this -- try k=5 as a starting
    # point (bigger k = the downtown shrinks faster as you move outward).
    intensity = np.exp(-dist * 5)

    # TODO: decide which cells are buildings. A cell becomes a building
    # when a random roll in [0, 1) is LESS than (intensity * density).
    # Hint: rng.uniform(size=(size, size)) gives one random roll per cell.
    is_building = rng.uniform(size=(size, size)) < (intensity * density)

    # building heights: random between 1 and 8 wherever there's a building
    building_height = np.where(is_building, rng.uniform(1, 8, size=(size, size)), 0.0)
    land_use = np.where(is_building, LAND_BUILDING, LAND_EMPTY)

    return building_height.astype(np.float32), land_use.astype(np.uint8)

In [67]:
# --- check yourself ---
_bh, _lu = generate_buildings(SIZE, density=0.6, seed=0)
assert _bh.shape == (SIZE, SIZE) and _lu.shape == (SIZE, SIZE)
_building_pct = (_lu == LAND_BUILDING).mean() * 100
assert 5 < _building_pct < 60, f"building coverage looks off ({_building_pct:.1f}%) -- check your intensity/is_building formulas"
# buildings should be concentrated near the center -- check the very corner is mostly empty
_corner_building_pct = (_lu[:20, :20] == LAND_BUILDING).mean() * 100
assert _corner_building_pct < _building_pct, "corners have as many buildings as the average -- intensity might not be decaying with distance"
print(f"Looks good! {_building_pct:.1f}% of the grid is buildings, concentrated near the center.")

Looks good! 12.0% of the grid is buildings, concentrated near the center.


## Step 3: Green space

Parks and green space should fill in some of the ground that's NOT already a
building — we don't want parks overlapping buildings.

The idea:
1. Find cells that are currently empty.
2. Randomly turn some fraction (`green_amount`) of just those cells into
   green space.


In [70]:
def add_green_space(land_use, green_amount=0.2, seed=None):
    """
    Takes an existing land_use array and returns a NEW one with some empty
    cells turned into green space. Does not touch existing buildings.

    TODO: fill in `empty_mask` and `green_mask` below.
    """
    rng = np.random.default_rng(seed)
    size = land_use.shape[0]

    # TODO: find cells that are currently empty (not a building).
    # Hint: this is a boolean array -- True where land_use == LAND_EMPTY.
    empty_mask = land_use == LAND_EMPTY

    # TODO: of the empty cells, randomly select roughly `green_amount`
    # fraction of them to become green space.
    # Hint: empty_mask & (rng.uniform(size=(size, size)) < green_amount)
    green_mask = empty_mask & (rng.uniform(size=(size, size)) < green_amount)
    land_use = land_use.copy()
    land_use[green_mask] = LAND_GREEN
    return land_use

In [15]:
# --- check yourself ---
_bh, _lu = generate_buildings(SIZE, density=0.6, seed=0)
_lu_before_pct = (_lu == LAND_BUILDING).mean() * 100
_lu_green = add_green_space(_lu, green_amount=0.3, seed=1)
_lu_after_building_pct = (_lu_green == LAND_BUILDING).mean() * 100
_green_pct = (_lu_green == LAND_GREEN).mean() * 100
assert abs(_lu_before_pct - _lu_after_building_pct) < 0.01, "building cells changed! add_green_space should only touch EMPTY cells"
assert _green_pct > 5, f"barely any green space showed up ({_green_pct:.1f}%) -- check your empty_mask/green_mask"
print(f"Looks good! green space = {_green_pct:.1f}%, buildings unchanged at {_lu_after_building_pct:.1f}%")

NameError: name 'generate_buildings' is not defined

## Step 4: Put it all together

Nothing to fill in here — just calling the three functions you just built.

In [ ]:
SEED = 42

terrain_height = generate_flat_terrain(SIZE, seed=SEED)
building_height, land_use = generate_buildings(SIZE, density=0.6, seed=SEED + 1)
land_use = add_green_space(land_use, green_amount=0.25, seed=SEED + 2)

print("Generated a city!")
print(f"  buildings: {(land_use == LAND_BUILDING).mean()*100:.1f}%")
print(f"  green space: {(land_use == LAND_GREEN).mean()*100:.1f}%")
print(f"  empty ground: {(land_use == LAND_EMPTY).mean()*100:.1f}%")

## Step 5: Visualize

This is the same 3-panel preview style the full project uses (terrain
height / building height / land use) — no water panel this week since flat
terrain has none. Nothing to fill in here, just run it and look at your city.


In [ ]:
LAND_USE_COLORS = {
    LAND_EMPTY: (0.85, 0.85, 0.8),     # light tan
    LAND_BUILDING: (0.75, 0.2, 0.2),   # red
    LAND_GREEN: (0.25, 0.6, 0.25),     # green
}

def land_use_rgb(land_use):
    rgb = np.zeros((*land_use.shape, 3), dtype=np.float32)
    for code_val, color in LAND_USE_COLORS.items():
        rgb[land_use == code_val] = color
    return rgb

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(terrain_height, cmap="terrain")
axes[0].set_title("Terrain height")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(building_height, cmap="magma")
axes[1].set_title("Building height")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

axes[2].imshow(land_use_rgb(land_use))
axes[2].set_title("Land use\n(red=building, green=park, tan=empty)")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"Week 1 city preview ({SIZE}x{SIZE})")
fig.tight_layout()
plt.show()

## Try it yourself

Now that everything runs, play with it:

- Change `density` in `generate_buildings(...)` (try `0.2` vs `0.9`) — how
  does the downtown change?
- Change `green_amount` in `add_green_space(...)`.
- Change `SEED` to get a different random layout each time.
- Try changing `k=5` inside your `intensity` formula (Step 2) — what happens
  with `k=2`? With `k=15`?

## What's coming in later weeks

This week was deliberately narrow so everyone gets a working result on day
one. Coming up:

- Multiple terrain types (hills, mountains, coastlines, rivers, peninsulas)
- City archetypes (polycentric, linear) and multiple building clusters
- Roads (with realistic gaps, not a perfect grid)
- Exporting all of this into real 3D geometry in Blender
- A trained model that reads a plain-English description and picks all of
  these parameters automatically

Everything you wrote today — the heightmap idea, the distance-based
intensity trick, the land-use grid — is the same core structure the final
pipeline uses. Later weeks add more parameters and more archetypes, but you
already understand the foundation.


## Questions?

Bring any questions or blockers to next session. Make sure your notebook runs top-to-bottom without errors before then.